# 01 — Análise exploratória do PaySim
> **LAB-VET-01 — estudo com PaySim (dados sintéticos).** Não use dados confidenciais, credenciais ou identificadores reais neste repositório. Execute as células em ordem. Os resultados históricos estão documentados separadamente; as saídas foram removidas para publicação.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Coloque o CSV público PaySim em data/ (não versionar arquivos de dados).
DATA_PATH = Path("data/PS_20174392719_1491204439457_log.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset ausente: {DATA_PATH.resolve()}. Consulte o README.")
df = pd.read_csv(DATA_PATH)
print("Dataset:", df.shape)

## Qualidade, distribuição e comportamento temporal

In [ ]:
# Soma total da coluna amount no DataFrame principal
total_amount = df["amount"].sum()
print(f"Valor total: {total_amount:,.2f}")

In [ ]:
print('Informações do dataset:')
df.info()

print('\nVALORES AUSENTES')
display(df.isna().sum().to_frame('ausentes'))

print('\nREGISTROS DUPLICADOS')
print(df.duplicated().sum())

print('\nTIPOS DE TRANSAÇÃO')
display(df['type'].value_counts())

print('\nDISTRIBUIÇÃO DE CLASSES')
display(df['isFraud'].value_counts())

In [ ]:
total = len(df)
fraudes = int(df['isFraud'].sum())
legitimas = total - fraudes

taxa_fraudes = fraudes / total * 100

print(f'\nTotal de transações: {total:,}')
print(f'Transações legítimas: {legitimas:,}')
print(f'Transações fraudulentas: {fraudes:,}')  
print(f'Taxa de fraudes: {taxa_fraudes:.4f}%')

In [ ]:
resumo = (
    df.groupby('type')
    .agg(
        total_transacoes=('isFraud', 'size'),
        fraudes=('isFraud', 'sum'),
        volume_financeiro=('amount', 'sum'),
    )
    .reset_index()
)

resumo['taxa_fraudes_pct'] =(resumo['fraudes'] / resumo['total_transacoes'] * 100
)
display(resumo.sort_values('fraudes', ascending=False))

In [ ]:
fraudes_por_step = (
    df.groupby("step")["isFraud"]
      .sum()
      .reset_index(name="fraudes")
)

plt.figure(figsize=(12, 4))

plt.plot(
    fraudes_por_step["step"],
    fraudes_por_step["fraudes"]
)

plt.title("Fraudes por etapa temporal — PaySim")
plt.xlabel("Step (hora simulada)")
plt.ylabel("Quantidade de fraudes")
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# Sprint 01 — Auditoria temporal

print("Primeiro step:", df["step"].min())
print("Último step:", df["step"].max())

print("\nDistribuição das fraudes por step:")

fraudes_tempo = (
    df.groupby("step")
      .agg(
          transacoes=("isFraud", "size"),
          fraudes=("isFraud", "sum")
      )
      .reset_index()
)

fraudes_tempo["taxa_fraude_pct"] = (
    fraudes_tempo["fraudes"] /
    fraudes_tempo["transacoes"] * 100
)

display(fraudes_tempo.head(10))
display(fraudes_tempo.tail(10))

print(
    "\nSteps com fraude:",
    (fraudes_tempo["fraudes"] > 0).sum()
)

In [ ]:
# Sprint 01 — Distribuição temporal por blocos

import numpy as np
import pandas as pd

limites = [0, 100, 200, 300, 400, 500, 600, 700, 743]

df["faixa_temporal"] = pd.cut(
    df["step"],
    bins=limites,
    include_lowest=True
)

resumo_temporal = (
    df.groupby("faixa_temporal", observed=True)
      .agg(
          transacoes=("isFraud", "size"),
          fraudes=("isFraud", "sum")
      )
      .reset_index()
)

resumo_temporal["legitimas"] = (
    resumo_temporal["transacoes"] -
    resumo_temporal["fraudes"]
)

resumo_temporal["taxa_fraude_pct"] = (
    resumo_temporal["fraudes"] /
    resumo_temporal["transacoes"] * 100
)

display(resumo_temporal)

**Interpretação:** dataset sintético, fortemente desbalanceado; distribuições temporais mudam entre períodos.

---
**Autoria**

```text
Alexandre Lotti
<!LOTTI>
```
